In [ ]:
import { logError } from "./logError.js";
import { reloadEnv } from "./reloadEnv.js";
await reloadEnv();
const token = Deno.env.get("OST_TOKEN");
const aosEndPoint = 'https://aos.adobe.io';
const wcsEndPoint = 'https://www.adobe.com';    
const api_key = 'wcms-commerce-ims-user-prod';
const environment = 'PROD'; 
const landscape = 'DRAFT'; // 'PUBLISHED';

const planTypesToTest = ['M2M', 'ABM', 'PUF'];
const countriesToTest = ['US', 'JP'];

const planTypes = {
  M2M: {
    commitment: 'MONTH',
    term: 'MONTHLY',
  },
  ABM: {
    commitment: 'YEAR',
    term: 'MONTHLY',
  },
  PUF: {
    commitment: 'YEAR',
    term: 'ANNUAL',
  }
};

In [ ]:
// Read a CSV file
const csv = await Deno.readTextFile('offers.csv');
const rows = csv.split('\n');
const headers = rows[0].split(',');
const data = rows.slice(1).map(row => row.split(','));
const offerData = data.map(row => {
  const offer = {};
  headers.forEach((header, index) => {
    offer[header] = row[index];
  });
  return offer;
});
console.log(offerData);


In [ ]:
// loop through offerData and call the AOS API for each offer
for (const offer of offerData) {
  const offerId = offer.OFFER_ID;
  const params = {
    environment,
    landscape,
    api_key,
  };
  const queryString = new URLSearchParams(params).toString();
  const url = new URL(`offers/${offerId}?${queryString}`, aosEndPoint).toString();
  const response = await fetch(url, {
    headers: {
      'X-Api-Key': api_key
    }
  });
  
  if (!response.ok) {
    await logError(response);
    continue;
  } 

  offer.details = await response.json();
  if (offer.details.length > 1) {
    console.log(`${offer.OFFER_ID} has multiple offers`);
  } else if (offer.details.length == 0) {
    console.log(`${offer.OFFER_ID} has no offers`);
  } else {
    offer.details = offer.details[0];
    console.log(JSON.stringify(offer.details, null, 2));
  }
}


In [ ]:
// loop through offerData and call the AOS API for each offer to get offer selector ID
for (const entry of offerData) {
  for (const planType of planTypesToTest) {
    const offer = {...entry.details, ...planTypes[planType]};
    entry[planType] = {};
    
    const params = {
      api_key
    }
    const body = {
      buying_program: offer.buying_program,
      commitment: offer.commitment,
      customer_segment: offer.customer_segment,
      market_segment: offer.market_segments[0],
      merchant: offer.merchant,
      offer_type: offer.offer_type,
      price_point: offer.price_point,
      product_arrangement_code: offer.product_arrangement_code,
      sales_channel: offer.sales_channel,
      term: offer.term,
    }

    const queryString = new URLSearchParams(params).toString();
    const url = new URL(`offer_selectors?${queryString}`, aosEndPoint).toString();
    
    const response = await fetch(url, {
      method: 'POST',
      headers: {
        'X-Api-Key': api_key,
        Authorization: `Bearer ${token}`,
        'Content-Type': 'application/json',
      },
      body: JSON.stringify(body),
    });
    
    if (!response.ok) {
      await logError(response);
      console.log('Error getting offer selector ID');
    }
    
    entry[planType].offerSelector = await response.json();
    console.log(entry[planType].offerSelector);
  }
}

In [ ]:
async function getAosPrice(offer, country, planType) {
  const params = {
    arrangement_code: offer.product_arrangement_code,
    buying_program: offer.buying_program,
    country,
    customer_segment: offer.customer_segment,
    language: offer.language,
    market_segment: offer.market_segments[0],
    merchant: offer.merchant,
    offer_type: offer.offer_type,
    price_point: offer.price_point,
    sales_channel: offer.sales_channel,
    commitment: planTypes[planType].commitment,
    term: planTypes[planType].term,
    api_key,
    environment: 'PROD',
    landscape,
    service_providers: 'PRICING',
    page: 0,
    page_size: 1000
  };
  const queryString = new URLSearchParams(params).toString();
  const url = new URL(`offers?${queryString}`, aosEndPoint).toString();

  const response = await fetch(url, {
    headers: {
      'X-Api-Key': api_key
    }
  });

  if (!response.ok) {
    await logError(response);
    return null;
  } 

  const details = await response.json();

  if (details.length > 1) {
    console.log(`${offer.offer_id} has multiple offers`);
    return null;
  } else if (details.length == 0) {
    console.log(`${offer.offer_id} has no offers`);
    return null;
  } else {
    return details[0].pricing;
  }
}

async function getWcsOffer(offerSelector, country) {
  const params = {
    offer_selector_ids: offerSelector.id,
    country,
    //locale: 'ja_JP',
    landscape,
    api_key: 'wcms-commerce-ims-ro-user-milo',
    language: 'MULT'
  };
  const queryString = new URLSearchParams(params).toString();
  const url = new URL(`web_commerce_artifact?${queryString}`, 'https://www.adobe.com').toString();
  
  const response = await fetch(url);
  
  if (!response.ok) {
    await logError(response);
    throw new Error('Request failed!');
  } 
  
  const artifact = await response.json();

  if (artifact.resolvedOffers.length > 1) {
    console.log(`${offerSelector.id} has multiple offers`);
    return null;
  } else if (artifact.resolvedOffers.length == 0) {
    console.log(`${offerSelector.id} has no offers`);
    return null;
  } else {
    return artifact.resolvedOffers[0];
  }
}

const results = [];
for (const entry of offerData) {
  for (const planType of planTypesToTest) {
    const offer = entry.details;
    const offerSelector = entry[planType].offerSelector;
    let countries = countriesToTest;
    if (countries === 'ALL') {
      countries = offer.countries;
    }

    // loop through offer.countries and call the WCS API for each country
    for (const country of countries) {

      const aosPrice = await getAosPrice(offer, country, planType);

      let aosPriceDisplay;
      if (!aosPrice) {
        console.log(`${offer.offer_id} ${country} ${planType} has no AOS price`);
        continue;
      } else if (aosPrice.prices.length > 1) {
        console.log(`${offer.offer_id} ${country} ${planType} has multiple AOS prices`);
        continue;
      } else {
        aosPriceDisplay = aosPrice.prices[0].price_details.display_rules.price;
      }
      
      let wcsPriceDisplay;
      const wcsOffer = await getWcsOffer(offerSelector, country);
      const wcsPrice = wcsOffer.priceDetails;
      wcsPriceDisplay = wcsPrice.price;

      const result = {
        offer_id: offer.offer_id,
        osi: offerSelector.id,
        aos_price: aosPriceDisplay,
        wcs_price: wcsPriceDisplay,
        plan_type: planType,
        country,
        product_arrangement_code: offer.product_arrangement_code,
        start_date: wcsOffer.startDate,
        end_date: wcsOffer.endDate,
        status: aosPriceDisplay === wcsPriceDisplay ? 'PASS' : 'FAIL'
      };
      results.push(result);
    }
  }
}

// Output results to a CSV file
const csvHeader = Object.keys(results[0]).join(',');
const csv = results.map(result => Object.values(result).join(',')).join('\n');
await Deno.writeTextFile('results.csv', csvHeader + '\n' + csv);
